# Izazov: Analiza teksta o znanosti o podacima

U ovom primjeru, napravimo jednostavnu vježbu koja pokriva sve korake tradicionalnog procesa znanosti o podacima. Ne morate pisati nikakav kod, možete samo kliknuti na ćelije ispod da ih izvršite i promatrate rezultat. Kao izazov, potičemo vas da isprobate ovaj kod s različitim podacima.

## Cilj

U ovoj lekciji razgovarali smo o različitim konceptima vezanim uz znanost o podacima. Pokušajmo otkriti još srodnih pojmova radeći **rudarenje teksta**. Počet ćemo s tekstom o znanosti o podacima, izdvojiti ključne riječi iz njega, a zatim pokušati vizualizirati rezultat.

Kao tekst, koristit ću stranicu o znanosti o podacima s Wikipedije:


In [ ]:
url = 'https://en.wikipedia.org/wiki/Data_science'

## Korak 1: Dohvaćanje podataka

Prvi korak u svakom procesu znanosti o podacima je dohvaćanje podataka. Za to ćemo koristiti knjižnicu `requests`:


In [ ]:
import requests

# Define a custom header.
headers = {
    'User-Agent': 'DataScienceChallenge/1.0 (myemail@gmail.com)'
}

# Pass the headers into the get request
response = requests.get(url, headers=headers)

if response.status_code == 200:
    text = response.content.decode('utf-8')
    print(text[:1000])
else:
    print(f"Error: {response.status_code}")

## Korak 2: Transformacija podataka

Sljedeći korak je pretvoriti podatke u oblik pogodan za obradu. U našem slučaju, preuzeli smo HTML izvorni kod sa stranice i potrebno ga je pretvoriti u običan tekst.

Postoji mnogo načina kako se to može napraviti. Koristit ćemo [BeautifulSoup](https://www.crummy.com/software/BeautifulSoup/), popularnu Python biblioteku za parsiranje HTML-a. BeautifulSoup nam omogućuje ciljanje specifičnih HTML elemenata, tako da se možemo fokusirati na glavni sadržaj članka sa Wikipedije i smanjiti neke navigacijske izbornike, bočne trake, podnožja i drugi irelevantni sadržaj (iako neki standardni tekstovi mogu i dalje ostati).


Prvo, trebamo instalirati BeautifulSoup biblioteku za razlaganje HTML-a:


In [ ]:
import sys
!{sys.executable} -m pip install beautifulsoup4

In [ ]:
from bs4 import BeautifulSoup

# Parse the HTML content
soup = BeautifulSoup(text, 'html.parser')

# Extract only the main article content from Wikipedia
# Wikipedia uses 'mw-parser-output' class for the main article content
content = soup.find('div', class_='mw-parser-output')

def clean_wikipedia_content(content_node):
    """Remove common non-article elements from a Wikipedia content node."""
    # Strip jump links, navboxes, reference lists/superscripts, edit sections, TOC, sidebars, etc.
    selectors = [
        '.mw-jump-link',
        '.navbox',
        '.reflist',
        'sup.reference',
        '.mw-editsection',
        '.hatnote',
        '.metadata',
        '.infobox',
        '#toc',
        '.toc',
        '.sidebar',
    ]
    for selector in selectors:
        for el in content_node.select(selector):
            el.decompose()

if content:
    # Clean the content node to better approximate article text only.
    clean_wikipedia_content(content)
    text = content.get_text(separator=' ', strip=True)
    print(text[:1000])
else:
    print("Could not find main content. Using full page text.")
    text = soup.get_text(separator=' ', strip=True)
    print(text[:1000])

## Korak 3: Dobivanje uvida

Najvažniji korak je pretvoriti naše podatke u neki oblik iz kojeg možemo izvući uvide. U našem slučaju želimo izvući ključne riječi iz teksta i vidjeti koje su ključne riječi smislenije.

Koristit ćemo Python biblioteku pod nazivom [RAKE](https://github.com/aneesha/RAKE) za izdvajanje ključnih riječi. Prvo, instalirajmo ovu biblioteku u slučaju da nije prisutna:


In [ ]:
import sys
!{sys.executable} -m pip install nlp_rake

Glavna funkcionalnost dostupna je iz objekta `Rake`, koji možemo prilagoditi pomoću nekih parametara. U našem slučaju, postavit ćemo minimalnu duljinu ključne riječi na 5 znakova, minimalnu učestalost ključne riječi u dokumentu na 3, te maksimalan broj riječi u ključnoj riječi na 2. Slobodno eksperimentirajte s drugim vrijednostima i promatrajte rezultat.


In [ ]:
import nlp_rake
extractor = nlp_rake.Rake(max_words=2,min_freq=3,min_chars=5)
res = extractor.apply(text)
res


Dobili smo popis pojmova zajedno s pripadajućom razinom važnosti. Kao što vidite, najrelevantnije discipline, poput strojnog učenja i velikih podataka, prisutne su na vrhu popisa.

## Korak 4: Vizualizacija rezultata

Ljudi najbolje mogu protumačiti podatke u vizualnom obliku. Stoga često ima smisla vizualizirati podatke kako bismo izvukli neke uvide. Možemo koristiti biblioteku `matplotlib` u Pythonu za jednostavan prikaz distribucije ključnih riječi s njihovom relevantnošću:


In [ ]:
import matplotlib.pyplot as plt

def plot(pair_list):
    k,v = zip(*pair_list)
    plt.bar(range(len(k)),v)
    plt.xticks(range(len(k)),k,rotation='vertical')
    plt.show()

plot(res)

Međutim, postoji još bolji način za vizualizaciju učestalosti riječi - korištenjem **Word Cloud**. Morat ćemo instalirati još jednu biblioteku za iscrtavanje oblaka riječi iz naše liste ključnih riječi.


In [ ]:
!{sys.executable} -m pip install wordcloud

Objekt `WordCloud` odgovoran je za primanje izvornog teksta ili unaprijed izračunatog popisa riječi s njihovim učestalostima, i vraća sliku, koja se zatim može prikazati pomoću `matplotlib`:


In [ ]:
from wordcloud import WordCloud
import matplotlib.pyplot as plt

wc = WordCloud(background_color='white',width=800,height=600)
plt.figure(figsize=(15,7))
plt.imshow(wc.generate_from_frequencies({ k:v for k,v in res }))

Također možemo proslijediti izvorni tekst u `WordCloud` - pogledajmo možemo li dobiti sličan rezultat:


In [ ]:
plt.figure(figsize=(15,7))
plt.imshow(wc.generate(text))

In [ ]:
wc.generate(text).to_file('images/ds_wordcloud.png')

Možete vidjeti da oblak riječi sada izgleda impresivnije, ali također sadrži puno šuma (npr. nepovezane riječi poput `Retrieved on`). Također, imamo manje ključnih riječi koje se sastoje od dvije riječi, poput *data scientist* ili *computer science*. To je zato što RAKE algoritam puno bolje odabire dobre ključne riječi iz teksta. Ovaj primjer ilustrira važnost prethodne obrade i čišćenja podataka, jer nam jasna slika na kraju omogućuje donošenje boljih odluka.

U ovoj vježbi prošli smo kroz jednostavan proces izdvajanja značenja iz Wikipedijinog teksta, u obliku ključnih riječi i oblaka riječi. Ovaj primjer je prilično jednostavan, ali dobro prikazuje sve tipične korake koje data scientist poduzima pri radu s podacima, počevši od nabave podataka do vizualizacije.

Na našem tečaju ćemo detaljno raspraviti sve te korake.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Napomena**:
Ovaj dokument je preveden korištenjem AI prevoditeljskog servisa [Co-op Translator](https://github.com/Azure/co-op-translator). Iako težimo točnosti, imajte na umu da automatski prijevodi mogu sadržavati greške ili netočnosti. Izvorni dokument na izvornom jeziku treba smatrati autoritativnim izvorom. Za važne informacije preporuča se profesionalni ljudski prijevod. Nismo odgovorni za bilo kakva nesporazumevanja ili pogrešne interpretacije koje proizlaze iz korištenja ovog prijevoda.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
